In [ ]:
import folium
import geojson
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import plotly.express as px
import plotly.graph_objects as go

from folium import plugins
from folium.plugins import HeatMap
from matplotlib.ticker import FuncFormatter, MultipleLocator
from scipy import stats

In [ ]:
df_rent = pd.read_csv('data/mean_rents_all_years.csv')
df_rent['Borough'] = df_rent['Borough'].str.upper()
df_rent.head()

In [ ]:
# df_rent[df_rent['Borough']=='STATEN ISLAND']

In [ ]:
# import glob
# import pandas as pd

# # Function to load and combine all boroughs for a given year
# def load_ethnicity_year(year):
#     files = glob.glob(f'data/ethnicity_{year}_*.csv')
#     dfs = []
    
#     for file in files:
#         # Extract borough name from filename
#         borough = file.split('_')[-1].replace('.csv', '')
        
#         # Read CSV
#         df = pd.read_csv(file, skiprows=5)
        
#         # Keep only the columns you need
#         df = df[['Unnamed: 0', 'Number.1', 'Percent.1']].copy()
        
#         # Rename columns and add borough
#         df.columns = ['ethnicity', 'number', 'percent']
#         if borough == 'island':
#             df['borough'] = 'STATEN ISLAND'
#         else:
#             df['borough'] = borough.upper()
        
#         dfs.append(df)
    
#     # Combine all boroughs
#     result = pd.concat(dfs, ignore_index=True)
#     return result

# # Load both years
# ethnicity_2010 = load_ethnicity_year(2010)
# ethnicity_2020 = load_ethnicity_year(2020)

# print(ethnicity_2010.head(10))
# print(ethnicity_2020.head(10))

In [ ]:
import requests
import pandas as pd

def get_race_data(year):
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": "NAME,B02001_001E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E",
        "for": "county:*",
        "in": "state:36"  # New York State
    }
    response = requests.get(url, params=params).json()
    df = pd.DataFrame(response[1:], columns=response[0])
    return df

race_2010 = get_race_data(2010)
race_2020 = get_race_data(2020)


In [ ]:
county_to_borough = {
    "Bronx County, New York": "BRONX",
    "Kings County, New York": "BROOKLYN",
    "New York County, New York": "MANHATTAN",
    "Queens County, New York": "QUEENS",
    "Richmond County, New York": "STATEN ISLAND"
}


In [ ]:
def clean_race(df):
    df['Borough'] = df['NAME'].map(county_to_borough)
    df = df[df['Borough'].notna()].copy()

    race_cols = [
        "B02001_001E","B02001_002E","B02001_003E",
        "B02001_004E","B02001_005E","B02001_006E","B02001_007E"
    ]

    for col in race_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

race_2010 = clean_race(race_2010)
race_2020 = clean_race(race_2020)


In [ ]:
race_cols = [
    "B02001_001E",  # total
    "B02001_002E",  # white
    "B02001_003E",  # black
    "B02001_004E",  # native
    "B02001_005E",  # asian
    "B02001_006E",  # pacific islander
    "B02001_007E"   # some other race
]

for col in race_cols:
    race_2010[col] = pd.to_numeric(race_2010[col], errors="coerce")
    race_2020[col] = pd.to_numeric(race_2020[col], errors="coerce")


In [ ]:
race_2010.head()

In [ ]:
def clean_ethnicity(df):
    df = df[df['number'].notna()].copy()
    
    # Debug: check what type number is
    print(f"Number dtype before: {df['number'].dtype}")
    print(f"Sample numbers: {df['number'].head()}")
    
    # Convert number column - remove commas first if they're strings
    df['number'] = df['number'].astype(str).str.replace(',', '')
    df['number'] = pd.to_numeric(df['number'], errors='coerce')
    
    print(f"Number dtype after: {df['number'].dtype}")
    print(f"Sample numbers after conversion: {df['number'].head()}")
    
    # Clean ethnicity and Borough names
    df['ethnicity'] = df['ethnicity'].str.strip()
    df['Borough'] = df['Borough'].str.strip().str.upper()
    
    # Get total population for each Borough
    total_pop = df[df['ethnicity'] == 'Total Population'].set_index('Borough')['number']
    print(f"\nTotal pop by Borough:\n{total_pop}")
    
    # Calculate percentage from total population
    df['percent_calc'] = df.apply(
        lambda row: (row['number'] / total_pop.get(row['Borough'])) * 100 
        if row['Borough'] in total_pop.index and pd.notna(row['number']) 
        else np.nan,
        axis=1
    )
    
    return df

In [ ]:
race_2010 = race_2010.rename(columns={
    "B02001_001E": "total_2010",
    "B02001_002E": "white_2010",
    "B02001_003E": "black_2010",
    "B02001_004E": "native_american_2010",
    "B02001_005E": "asian_2010",
    "B02001_006E": "pacific_islander_2010",
    "B02001_007E": "other_2010"
})

race_2020 = race_2020.rename(columns={
    "B02001_001E": "total_2020",
    "B02001_002E": "white_2020",
    "B02001_003E": "black_2020",
    "B02001_004E": "native_american_2020",
    "B02001_005E": "asian_2020",
    "B02001_006E": "pacific_islander_2020",
    "B02001_007E": "other_2020"
})


In [ ]:
# list(ethnicity_2010['ethnicity'].unique())

In [ ]:
# eth10 = clean_ethnicity(ethnicity_2010)
# eth20 = clean_ethnicity(ethnicity_2020)


In [ ]:
race = race_2010.merge(race_2020, on="Borough", suffixes=("_2010", "_2020"))

In [ ]:
race.head()

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import pearsonr

# ---------------------------------------------------------
# 0. Borough colors (edit these hex codes anytime)
# ---------------------------------------------------------

borough_colors = {
    "MANHATTAN": "#1f77b4",
    "BROOKLYN": "#ff7f0e",
    "QUEENS": "#2ca02c",
    "BRONX": "#d62728",
    "STATEN ISLAND": "#9467bd"
}

# ---------------------------------------------------------
# 1. Your existing rent pipeline (unchanged)
# ---------------------------------------------------------

rent_cols_2010 = [c for c in df_rent.columns if c.startswith("2010")]
rent_cols_2020 = [c for c in df_rent.columns if c.startswith("2020")]

df_rent['rent_2010_mean'] = df_rent[rent_cols_2010].mean(axis=1)
df_rent['rent_2020_mean'] = df_rent[rent_cols_2020].mean(axis=1)

rent_2010 = df_rent[['Borough', 'apartmentType', 'rent_2010_mean']].copy()
rent_2010['year'] = "2010"
rent_2010 = rent_2010.rename(columns={'rent_2010_mean': 'rent'})

rent_2020 = df_rent[['Borough', 'apartmentType', 'rent_2020_mean']].copy()
rent_2020['year'] = "2020"
rent_2020 = rent_2020.rename(columns={'rent_2020_mean': 'rent'})

rent_long = pd.concat([rent_2010, rent_2020], ignore_index=True)

# ---------------------------------------------------------
# 2. Compute race shares (distribution)
# ---------------------------------------------------------

for group in ["white", "black", "asian", "native_american", "pacific_islander", "other"]:
    race[f"{group}_share_2010"] = race[f"{group}_2010"] / race["total_2010"]
    race[f"{group}_share_2020"] = race[f"{group}_2020"] / race["total_2020"]

# ---------------------------------------------------------
# 3. Reshape race shares into long format
# ---------------------------------------------------------

race_long = race.melt(
    id_vars=["Borough"],
    value_vars=[
        "white_share_2010", "black_share_2010", "asian_share_2010",
        "native_american_share_2010", "pacific_islander_share_2010", "other_share_2010",
        "white_share_2020", "black_share_2020", "asian_share_2020",
        "native_american_share_2020", "pacific_islander_share_2020", "other_share_2020"
    ],
    var_name="group_year",
    value_name="share"
)

race_long["group"] = race_long["group_year"].str.extract(r"(.*)_share")[0]
race_long["year"] = race_long["group_year"].str.extract(r"(\d{4})")[0]

# ---------------------------------------------------------
# 4. Merge race shares with rent_long
# ---------------------------------------------------------

df_plot = race_long.merge(rent_long, on=["Borough", "year"], how="left")

# ---------------------------------------------------------
# 5. Build interactive Plotly figure with apartment type dropdown
# ---------------------------------------------------------

apt_types = sorted(df_plot["apartmentType"].unique())
groups = df_plot["group"].unique()
years = ["2010", "2020"]

year_shapes = {"2010": "circle", "2020": "square"}

cols = 2
rows = int(np.ceil(len(groups) / cols))

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=groups,
    horizontal_spacing=0.12,
    vertical_spacing=0.12
)

# Visibility masks for apartment types
visibility_masks = {apt: [] for apt in apt_types}

row = 1
col = 1

for group in groups:
    for apt in apt_types:
        for year in years:
            sub = df_plot[
                (df_plot["group"] == group) &
                (df_plot["apartmentType"] == apt) &
                (df_plot["year"] == year)
            ]

            # Scatter
            scatter = go.Scatter(
                x=sub["share"],
                y=sub["rent"],
                mode="markers",
                marker=dict(
                    size=12,
                    symbol=year_shapes[year],
                    color=[borough_colors[b] for b in sub["Borough"]]
                ),
                name=f"{group} — {year}",
                text=sub["Borough"],
                hovertemplate="<b>%{text}</b><br>Share: %{x:.2%}<br>Rent: %{y}<br>"
            )
            fig.add_trace(scatter, row=row, col=col)
            
            # Add to visibility mask for each apartment type
            for a in apt_types:
                visibility_masks[a].append(a == apt)

    col += 1
    if col > cols:
        col = 1
        row += 1

# ---------------------------------------------------------
# 6. Dropdown menu (apartment type)
# ---------------------------------------------------------

buttons = []
for apt in apt_types:
    buttons.append(
        dict(
            label=apt,
            method="update",
            args=[{"visible": visibility_masks[apt]}]
        )
    )

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=1.05,
            y=1,
            showactive=True
        )
    ],
    height=300 * rows,
    width=900,
    title="Race Share vs Rent — 2010 & 2020",
    showlegend=False
)

# Set default visibility to first apartment type
default_apt = apt_types[0]
fig.update_traces(visible=False)
for i, vis in enumerate(visibility_masks[default_apt]):
    if i < len(fig.data):
        fig.data[i].visible = vis

fig.update_xaxes(title_text="Race Share")
fig.update_yaxes(title_text="Average Rent")

fig.show()

In [ ]:
race['white_share_2010'] = race['white_2010'] / race['B02001_001E_2010']
race['white_share_2020'] = race['white_2020'] / race['B02001_001E_2020']

race['black_share_2010'] = race['black_2010'] / race['B02001_001E_2010']
race['black_share_2020'] = race['black_2020'] / race['B02001_001E_2020']

race['asian_share_2010'] = race['asian_2010'] / race['B02001_001E_2010']
race['asian_share_2020'] = race['asian_2020'] / race['B02001_001E_2020']

race['native_share_2010'] = race['native_2010'] / race['B02001_001E_2010']
race['native_share_2020'] = race['native_2020'] / race['B02001_001E_2020']

race['pi_share_2010'] = race['pi_2010'] / race['B02001_001E_2010']
race['pi_share_2020'] = race['pi_2020'] / race['B02001_001E_2020']

race['other_share_2010'] = race['other_2010'] / race['B02001_001E_2010']
race['other_share_2020'] = race['other_2020'] / race['B02001_001E_2020']


In [ ]:
for group in ["white", "black", "native", "asian", "pi", "other"]:
    race[f"{group}_change_pct"] = (
        (race[f"{group}_2020"] - race[f"{group}_2010"]) / race[f"{group}_2010"]
    )


In [ ]:
for group in ["white", "black", "asian", "native", "pi", "other"]:
    race[f"{group}_share_change"] = (
        race[f"{group}_share_2020"] - race[f"{group}_share_2010"]
    )


In [ ]:
# eth20[eth20['ethnicity']=='Danish']

In [ ]:
race.dtypes


In [ ]:
import statsmodels.api as sm

# Merge race shares + rent means into one dataframe
merged = df_rent.merge(race, on="Borough", how="left")

# Compute rent change
merged["rent_change"] = merged["rent_2020_mean"] - merged["rent_2010_mean"]

# Compute share changes for each group
groups = ["white", "black", "asian", "native_american", "pacific_islander", "other"]

for g in groups:
    merged[f"{g}_share_change"] = merged[f"{g}_share_2020"] - merged[f"{g}_share_2010"]

In [ ]:
merged

In [ ]:

borough_colors = {
    "MANHATTAN": "#1f77b4",
    "BROOKLYN": "#ff7f0e",
    "QUEENS": "#2ca02c",
    "BRONX": "#d62728",
    "STATEN ISLAND": "#9467bd",
    "NEW YORK CITY": "#bd678e",
}
import plotly.graph_objects as go
from plotly.subplots import make_subplots

apt_type = "One bedroom"  # choose any apartment type

sub = merged[merged["apartmentType"] == apt_type]


fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=groups,
    horizontal_spacing=0.12,
    vertical_spacing=0.12
)

row, col = 1, 1

for g in groups:
    x2010 = sub[f"{g}_share_2010"]
    y2010 = sub["rent_2010_mean"]
    x2020 = sub[f"{g}_share_2020"]
    y2020 = sub["rent_2020_mean"]

    # 2010 points
    fig.add_trace(
        go.Scatter(
            x=x2010,
            y=y2010,
            mode="markers",
            marker=dict(
                size=10,
                symbol="circle",
                color=[borough_colors[b] for b in sub["Borough"]]
            ),
            text=sub["Borough"],
            showlegend=False
        ),
        row=row, col=col
    )

    # 2020 points
    fig.add_trace(
        go.Scatter(
            x=x2020,
            y=y2020,
            mode="markers",
            marker=dict(
                size=10,
                symbol="square",
                color=[borough_colors[b] for b in sub["Borough"]]
            ),
            text=sub["Borough"],
            showlegend=False
        ),
        row=row, col=col
    )

    # Arrows
    for i in range(len(sub)):
        fig.add_annotation(
            x=x2020.iloc[i], y=y2020.iloc[i],
            ax=x2010.iloc[i], ay=y2010.iloc[i],
            showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1
        )

    col += 1
    if col > 2:
        col = 1
        row += 1

fig.update_layout(
    height=900,
    width=900,
    title=f"Race share vs rent (2010 → 2020 arrows) — {apt_type}"
)

fig.show()


In [ ]:
race[race['Borough']=='BRONX']['white_2010'][0]/race[race['Borough']=='BRONX']['total_2010'][0]*100

In [ ]:
1365725-int(race[race['Borough']=='BRONX']['white_2010'][0] + race[race['Borough']=='BRONX']['black_2010'][0]+race[race['Borough']=='BRONX']['native_american_2010'][0] + race[race['Borough']=='BRONX']['asian_2010'][0]+race[race['Borough']=='BRONX']['pacific_islander_2010'][0]+race[race['Borough']=='BRONX']['other_2010'][0])

In [ ]:
race[race['Borough']=='BRONX']['white_2020'][0]/race[race['Borough']=='BRONX']['total_2020'][0]

In [ ]:
1365725-1322870

In [ ]:
race.head()

In [ ]:
import plotly.express as px

# Compute changes at borough × apartmentType × group level
rows = []
# Merge df_rent with race data on Borough
merged = df_rent.merge(race, on="Borough", how="left")

# Then loop through merged data
for g in groups:
    for _, row in merged.iterrows():
        b = row["Borough"]
        # All race columns are already in row now
        s10 = row[f"{g}_share_2010"]
        s20 = row[f"{g}_share_2020"]

        b = row["Borough"]
        apt = row["apartmentType"]

        r10 = row["rent_2010_mean"]
        r20 = row["rent_2020_mean"]
        drent = r20 - r10

        # rrow = race[race["Borough"] == b].iloc[0]
        # s10 = rrow[f"{g}_share_2010"]
        # s20 = rrow[f"{g}_share_2020"]
        dshare = s20 - s10

        rows.append({
            "Borough": b,
            "apartmentType": apt,
            "group": g,
            "rent_2010": r10,
            "rent_2020": r20,
            "rent_change": drent,
            "share_2010": s10,
            "share_2020": s20,
            "share_change": dshare
        })

df_change = pd.DataFrame(rows)

# Example: one apartment type at a time
apt_type = "One bedroom"
sub = df_change[df_change["apartmentType"] == apt_type]

fig = px.scatter(
    sub,
    x="share_change",
    y="rent_change",
    color="Borough",
    facet_col="group",
    facet_col_wrap=3,
    trendline="ols",
    labels={
        "share_change": "Change in race share (2020 − 2010)",
        "rent_change": "Change in average rent (2020 − 2010)"
    },
    title=f"Change in race share vs change in rent — {apt_type}"
)
fig.show()


In [ ]:
import plotly.express as px

apt_type = "One bedroom"
sub = merged[merged["apartmentType"] == apt_type]

fig = px.scatter(
    sub,
    x="white_share_change",   # you can facet by group below
    y="rent_change",
    color="Borough",
    facet_col="group",
    facet_col_wrap=3,
    labels={
        "white_share_change": "Change in race share",
        "rent_change": "Change in rent"
    },
    title=f"Change in race share vs change in rent — {apt_type}"
)

fig.show()


In [ ]:
! pip install statsmodels

In [ ]:



# Aggregate to borough × apartmentType (one row per combo)
agg_rows = []
for _, row_r in df_rent.iterrows():
    b = row_r["Borough"]
    apt = row_r["apartmentType"]

    r10 = row_r["rent_2010_mean"]
    r20 = row_r["rent_2020_mean"]
    drent = r20 - r10

    rrow = race[race["Borough"] == b].iloc[0]

    row_out = {
        "Borough": b,
        "apartmentType": apt,
        "rent_change": drent
    }

    for g in groups:
        row_out[f"{g}_share_change"] = (
            rrow[f"{g}_share_2020"] - rrow[f"{g}_share_2010"]
        )

    agg_rows.append(row_out)

df_model = pd.DataFrame(agg_rows)

# Example: model for one apartment type
apt_type = "One bedroom"
m_sub = df_model[df_model["apartmentType"] == apt_type].copy()

X = m_sub[[f"{g}_share_change" for g in groups]]
X = sm.add_constant(X)
y = m_sub["rent_change"]

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
# group_map = {

#     # -------------------------
#     # HISPANIC / LATINO
#     # -------------------------
#     "Hispanic or Latino (of any race)": "Hispanic",
#     "Mexican": "Mexican",
#     "Central American": "Central American",
#     "Costa Rican": "Central American",
#     "Guatemalan": "Central American",
#     "Honduran": "Central American",
#     "Nicaraguan": "Central American",
#     "Panamanian": "Central American",
#     "Salvadoran": "Central American",

#     "South American": "South American",
#     "Argentinean": "South American",
#     "Bolivian": "South American",
#     "Chilean": "South American",
#     "Colombian": "South American",
#     "Ecuadorian": "South American",
#     "Paraguayan": "South American",
#     "Peruvian": "South American",
#     "Uruguayan": "South American",
#     "Venezuelan": "South American",

#     "Caribbean Hispanic": "Caribbean Hispanic",
#     "Cuban": "Caribbean Hispanic",
#     "Dominican": "Caribbean Hispanic",
#     "Puerto Rican": "Caribbean Hispanic",

#     "Other Hispanic": "Other Hispanic",
#     "Spaniard": "Other Hispanic",
#     "Spanish": "Other Hispanic",
#     "Spanish American": "Other Hispanic",
#     "Garifuna": "Other Hispanic",

#     # -------------------------
#     # WHITE (NON-HISPANIC)
#     # -------------------------
#     "White": "White",
#     "European": "White",
#     "Albanian": "White",
#     "Armenian": "White",
#     "Austrian": "White",
#     "Azerbaijani": "White",
#     "Belarusian": "White",
#     "Belgian": "White",
#     "Bosnian and Herzegovinian": "White",
#     "British": "White",
#     "Bulgarian": "White",
#     "Croatian": "White",
#     "Cypriot": "White",
#     "Czech": "White",
#     "Danish": "White",
#     "Dutch": "White",
#     "English": "White",
#     "Estonian": "White",
#     "Finnish": "White",
#     "French": "White",
#     "Georgian": "White",
#     "German": "White",
#     "Greek": "White",
#     "Hungarian": "White",
#     "Irish": "White",
#     "Italian": "White",
#     "Kosovan": "White",
#     "Latvian": "White",
#     "Lithuanian": "White",
#     "Macedonian": "White",
#     "Maltese": "White",
#     "Moldovan": "White",
#     "Montenegrin": "White",
#     "Norwegian": "White",
#     "Polish": "White",
#     "Portuguese": "White",
#     "Romanian": "White",
#     "Russian": "White",
#     "Scandinavian": "White",
#     "Scots-Irish": "White",
#     "Scottish": "White",
#     "Serbian": "White",
#     "Slavic": "White",
#     "Slovak": "White",
#     "Slovenian": "White",
#     "Swedish": "White",
#     "Swiss": "White",
#     "Turkish": "White",
#     "Ukrainian": "White",
#     "Welsh": "White",

#     "Other White": "White",
#     "Australian": "White",
#     "Canadian": "White",
#     "French Canadian": "White",
#     "New Zealander": "White",

#     # -------------------------
#     # MIDDLE EASTERN / NORTH AFRICAN (Middle Eastern)
#     # -------------------------
#     "Middle Eastern or North African": "Middle Eastern",
#     "Algerian": "Middle Eastern",
#     "Arab": "Middle Eastern",
#     "Egyptian": "Middle Eastern",
#     "Iranian": "Middle Eastern",
#     "Iraqi": "Middle Eastern",
#     "Israeli": "Middle Eastern",
#     "Jordanian": "Middle Eastern",
#     "Lebanese": "Middle Eastern",
#     "Moroccan": "Middle Eastern",
#     "Palestinian": "Middle Eastern",
#     "Syrian": "Middle Eastern",
#     "Tunisian": "Middle Eastern",
#     "Yemeni": "Middle Eastern",

#     # -------------------------
#     # BLACK / AFRICAN AMERICAN
#     # -------------------------
#     "Black or African American": "Black",
#     "African American": "Black",

#     "Sub-Saharan African": "Black",
#     "Burkinabe": "Black",
#     "Cameroonian": "Black",
#     "Congolese": "Black",
#     "Ethiopian": "Black",
#     "Gambian": "Black",
#     "Ghanaian": "Black",
#     "Guinean": "Black",
#     "Ivoirian": "Black",
#     "Kenyan": "Black",
#     "Liberian": "Black",
#     "Malian": "Black",
#     "Nigerian (Nigeria)": "Black",
#     "Senegalese": "Black",
#     "Sierra Leonean": "Black",
#     "South African": "Black",
#     "Sudanese": "Black",
#     "Togolese": "Black",

#     "Caribbean": "Black",
#     "Antiguan and Barbudan": "Black",
#     "Bahamian": "Black",
#     "Barbadian": "Black",
#     "Dominica Islander": "Black",
#     "Grenadian": "Black",
#     "Haitian": "Black",
#     "Jamaican": "Black",
#     "Kittian and Nevisian": "Black",
#     "St. Lucian": "Black",
#     "Trinidadian and Tobagonian": "Black",
#     "U.S. Virgin Islander": "Black",
#     "Vincentian": "Black",
#     "West Indian": "Black",

#     "Other Black or African American": "Black",

#     # -------------------------
#     # AMERICAN INDIAN / ALASKA NATIVE
#     # -------------------------
#     "American Indian and Alaska Native": "Native American",
#     "Alaska Native": "Native American",
#     "American Indian": "Native American",
#     "Blackfeet Tribe of the Blackfeet Indian Reservation of Montana": "Native American",
#     "Cherokee": "Native American",
#     "Central American Indian (all tribes)": "Native American",
#     "Mexican Indian (all tribes)": "Native American",
#     "Aztec": "Native American",
#     "South American Indian (all tribes)": "Native American",
#     "Ecuadorian Indian": "Native American",
#     "Guyanese South American Indian": "Native American",
#     "Inca": "Native American",
#     "Caribbean Indian (all tribes)": "Native American",
#     "Taino": "Native American",
#     "Mesoamerican Indian (all tribes)": "Native American",
#     "Maya": "Native American",

#     # -------------------------
#     # ASIAN
#     # -------------------------
#     "Asian": "Asian",
#     "East Asian": "Asian",
#     "Chinese,  except Taiwanese": "Asian",
#     "Japanese": "Asian",
#     "Korean": "Asian",
#     "Taiwanese": "Asian",

#     "Central Asian": "Asian",
#     "Afghan": "Asian",
#     "Kazakh": "Asian",
#     "Kyrgyz": "Asian",
#     "Tajik": "Asian",
#     "Uzbek": "Asian",

#     "South Asian": "Asian",
#     "Asian Indian": "Asian",
#     "Bangladeshi": "Asian",
#     "Nepalese": "Asian",
#     "Pakistani": "Asian",
#     "Sikh": "Asian",
#     "Sri Lankan": "Asian",

#     "Southeast Asian": "Asian",
#     "Burmese": "Asian",
#     "Cambodian": "Asian",
#     "Filipino": "Asian",
#     "Indonesian": "Asian",
#     "Malaysian": "Asian",
#     "Singaporean": "Asian",
#     "Thai": "Asian",
#     "Vietnamese": "Asian",

#     "Other Asian": "Asian",

#     # -------------------------
#     # PACIFIC ISLANDER
#     # -------------------------
#     "Native Hawaiian and Other Pacific Islander": "Pacific Islander",
#     "Polynesian": "Pacific Islander",
#     "Native Hawaiian": "Pacific Islander",
#     "Samoan": "Pacific Islander",
#     "Micronesian": "Pacific Islander",
#     "Chamorro": "Pacific Islander",

#     # -------------------------
#     # SOME OTHER RACE
#     # -------------------------
#     "Some Other Race": "Some Other Race",
#     "Belizean": "Some Other Race",
#     "Brazilian": "Some Other Race",
#     "Guyanese": "Some Other Race",
# }


In [ ]:
# def collapse_groups(df):
#     df = df[df['ethnicity'].isin(group_map.keys())].copy()
#     df['group'] = df['ethnicity'].map(group_map)
#     grouped = df.groupby(['borough', 'group'])['number'].sum().reset_index()
#     return grouped

# eth10g = collapse_groups(eth10)
# eth20g = collapse_groups(eth20)


In [ ]:
import statsmodels.api as sm
import numpy as np

apt_type = "One bedroom"
m_sub = merged[merged["apartmentType"] == apt_type].copy()

X = m_sub[[f"{g}_share_change" for g in groups]]
X = sm.add_constant(X)
y = m_sub["rent_change"]

# Clean NaN and inf
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
y = y.replace([np.inf, -np.inf], np.nan).fillna(0)

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
# eth10g

In [ ]:
# sorted(eth10['ethnicity'].unique())


In [ ]:
# def collapse_groups(df):
#     df = df[df['ethnicity'].isin(group_map.keys())].copy()
#     df['group'] = df['ethnicity'].map(group_map)
#     grouped = df.groupby(['borough', 'group'])['percent_calc'].sum().reset_index()
#     grouped = grouped.rename(columns={'percent_calc': 'pct'})
#     return grouped


# eth10g = collapse_groups(eth10)
# eth20g = collapse_groups(eth20)


In [ ]:
# eth10g

In [ ]:
# eth_change = eth10g.merge(
#     eth20g,
#     on=['borough', 'group'],
#     suffixes=('_2010', '_2020')
# )


# eth_change['change_pct'] = (
#     (eth_change['pct_2020'] - eth_change['pct_2010']) / eth_change['pct_2010']
# )


In [ ]:
# race_long = race.melt(
#     id_vars=['Borough'],
#     value_vars=[
#         'white_change_pct',
#         'black_change_pct',
#         'native_change_pct',
#         'asian_change_pct',
#         'pi_change_pct',
#         'other_change_pct'
#     ],
#     var_name='group',
#     value_name='change_pct'
# )

race_long = race.melt(
    id_vars=['Borough'],
    value_vars=[
        'white_share_change',
        'black_share_change',
        'asian_share_change',
        'native_share_change',
        'pi_share_change',
        'other_share_change'
    ],
    var_name='group',
    value_name='change_pct'
)

race_long['group'] = race_long['group'].str.replace('_share_change', '')


In [ ]:
rent_cols_2010 = [c for c in df_rent.columns if c.startswith("2010")]
rent_cols_2020 = [c for c in df_rent.columns if c.startswith("2020")]

df_rent['rent_2010_mean'] = df_rent[rent_cols_2010].mean(axis=1)
df_rent['rent_2020_mean'] = df_rent[rent_cols_2020].mean(axis=1)

df_rent['rent_change_pct'] = (
    (df_rent['rent_2020_mean'] - df_rent['rent_2010_mean'])
    / df_rent['rent_2010_mean']
)


In [ ]:
rent_change = df_rent[['Borough', 'apartmentType', 'rent_change_pct']]


In [ ]:
rent_change.head()

In [ ]:
df_merged = race_long.merge(rent_change, on='Borough', how='left')


In [ ]:
df_merged.head()

In [ ]:
corr = (
    df_merged.groupby('group')[['change_pct', 'rent_change_pct']]
    .corr()
    .iloc[0::2, -1]
)

print(corr)


In [ ]:
# import plotly.express as px

# fig = px.scatter(
#     df_merged,
#     x="change_pct",
#     y="rent_change_pct",
#     color="group",
#     hover_name="borough",
#     hover_data={
#         "change_pct": ":.2%",
#         "rent_change_pct": ":.2%",
#         "group": True,
#         "borough": True
#     },
#     labels={
#         "change_pct": "Ethnicity % Change (2010 → 2020)",
#         "rent_change_pct": "Rent % Change (2010 → 2020)",
#         "group": "Ethnicity Group"
#     },
#     title="Correlation Between Ethnicity Change and Rent Change (2010–2020)"
# )

# fig.update_traces(marker=dict(size=14, opacity=0.8))

# fig.show()


In [ ]:
def safe_regression(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)

    # Remove NaN / inf
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    # Need at least 2 points
    if len(x) < 2:
        return None, None

    # Need variance in x
    if np.allclose(x, x[0]):
        return None, None

    # Compute slope and intercept manually
    x_mean = x.mean()
    y_mean = y.mean()

    m = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean)**2)
    b = y_mean - m * x_mean

    return m, b


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Get unique ethnicity groups
groups = df_merged['group'].unique()
n_groups = len(groups)

# Create subplot grid (auto layout: 2 columns)
cols = 2
rows = int(np.ceil(n_groups / cols))

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=groups,
    horizontal_spacing=0.12,
    vertical_spacing=0.12
)

row = 1
col = 1

for group in groups:
    sub = df_merged[df_merged['group'] == group]

    # Scatter points
    fig.add_trace(
        go.Scatter(
            x=sub['change_pct'],
            y=sub['rent_change_pct'],
            mode='markers',
            name=group,
            marker=dict(size=10),
            text=sub['Borough'],
            hovertemplate=(
                "<b>%{text}</b><br>" +
                "Ethnicity change: %{x:.2%}<br>" +
                "Rent change: %{y:.2%}<br>"
            )
        ),
        row=row, col=col
    )

    # Regression line
    x = sub['change_pct']
    y = sub['rent_change_pct']

    m, b = safe_regression(x, y)

    if m is not None:
        x_line = np.linspace(x.min(), x.max(), 50)
        y_line = m * x_line + b

        fig.add_trace(
            go.Scatter(
                x=x_line,
                y=y_line,
                mode='lines',
                line=dict(color='black', width=2),
                showlegend=False
            ),
            row=row, col=col
        )
    else:
        print(f"Skipping regression for {group} (insufficient data or variance)")

    # Move to next subplot
    col += 1
    if col > cols:
        col = 1
        row += 1


# Layout
fig.update_layout(
    height=300 * rows,
    width=900,
    title="Ethnicity Change vs Rent Change (2010–2020) — Per Ethnicity Group",
    showlegend=False
)

fig.update_xaxes(title_text="Ethnicity % Change")
fig.update_yaxes(title_text="Rent % Change")

fig.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy.stats import pearsonr

apt_types = df_merged['apartmentType'].unique()
groups = df_merged['group'].unique()

# Set default apartment type
default_apt = "One bedroom"

cols = 2
rows = int(np.ceil(len(groups) / cols))

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=groups,
    horizontal_spacing=0.12,
    vertical_spacing=0.12
)

# Store visibility masks for dropdown
visibility_masks = {apt: [] for apt in apt_types}

row = 1
col = 1

for group in groups:
    for apt in apt_types:
        sub = df_merged[(df_merged['group'] == group) & (df_merged['apartmentType'] == apt)]

        # Scatter
        scatter = go.Scatter(
            x=sub['change_pct'],
            y=sub['rent_change_pct'],
            mode='markers',
            name=f"{group} — {apt}",
            marker=dict(size=10),
            text=sub['Borough'],
            hovertemplate=(
                "<b>%{text}</b><br>" +
                "Ethnicity change: %{x:.2%}<br>" +
                "Rent change: %{y:.2%}<br>"
            )
        )
        fig.add_trace(scatter, row=row, col=col)

        # Regression + R² + correlation
        x = sub['change_pct']
        y = sub['rent_change_pct']
        m, b = safe_regression(x, y)

        if m is not None:
            x_line = np.linspace(x.min(), x.max(), 50)
            y_line = m * x_line + b

            # Compute correlation + R²
            r, _ = pearsonr(x, y)
            r2 = r ** 2

            reg = go.Scatter(
                x=x_line,
                y=y_line,
                mode='lines',
                line=dict(color='black', width=2),
                showlegend=False,
                hoverinfo='skip'
            )
            fig.add_trace(reg, row=row, col=col)

            # Add annotation for R² and r
            # Calculate subplot axis index
        else:
            # Dummy trace to keep indexing consistent
            reg = go.Scatter(x=[], y=[], showlegend=False)
            fig.add_trace(reg, row=row, col=col)

        # Visibility mask: 2 traces per apt type (scatter + regression)
        for a in apt_types:
            visibility_masks[a].extend([a == apt, a == apt])

    col += 1
    if col > cols:
        col = 1
        row += 1

# Dropdown menu
buttons = []
for apt in apt_types:
    buttons.append(
        dict(
            label=apt,
            method="update",
            args=[{"visible": visibility_masks[apt]}]
        )
    )

fig.update_layout(
    height=300 * rows,
    width=900,
    title="Ethnicity Change vs Rent Change (2010–2020) — Interactive by Apartment Type",
    showlegend=False,
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=1.05,
            y=1,
            showactive=True
        )
    ]
)

# Set default visibility to One bedroom
fig.update_traces(visible=False)
for i, vis in enumerate(visibility_masks[default_apt]):
    fig.data[i].visible = vis

fig.update_xaxes(title_text="Ethnicity % Change")
fig.update_yaxes(title_text="Rent % Change")

fig.show()


In [ ]:
temp =(df_rent[df_rent['Borough']=='STATEN ISLAND'])
temp

In [ ]:
temp.dropna(how='all', axis=1)

In [271]:
import pandas as pd
import numpy as np
import requests
import plotly.express as px
import plotly.graph_objects as go

df_rent_area = pd.read_csv('data/mean_rents_by_area.csv')
df_rent_area['Borough'] = df_rent_area['Borough'].str.upper()

rent_long = df_rent_area.melt(
    id_vars=["apartmentType", "Borough", "areaName"],
    var_name="month",
    value_name="rent"
)

rent_long["year"] = rent_long["month"].str.slice(0, 4)

rent_yearly = (
    rent_long
    .groupby(["Borough", "areaName", "year"], as_index=False)
    .agg(rent_mean=("rent", "mean"))
)


In [272]:
rent_yearly

,Borough,areaName,year,rent_mean
0,BRONX,Baychester,2010,NaN
1,BRONX,Baychester,2011,NaN
2,BRONX,Baychester,2012,NaN
3,BRONX,Baychester,2013,NaN
4,BRONX,Baychester,2014,NaN
...,...,...,...,...
3163,STATEN ISLAND,Staten Island,2021,2041.097222
3164,STATEN ISLAND,Staten Island,2022,1964.918605
3165,STATEN ISLAND,Staten Island,2023,2132.344444
3166,STATEN ISLAND,Staten Island,2024,2570.985714


In [ ]:
def load_acs(table, year):
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": "NAME," + ",".join([f"{table}_{i:03d}E" for i in range(1, 10)]),  # fewer variables
        "for": "tract:*",
        "in": "state:36 county:*"
    }
    r = requests.get(url, params=params)
    
    # # Debug: r
    
    data = r.json()
    return pd.DataFrame(data[1:], columns=data[0])

In [339]:
race2010 = load_acs("B02001", 2010)
race2020 = load_acs("B02001", 2020)

hisp2010 = load_acs("B03002", 2010)
hisp2020 = load_acs("B03002", 2020)


Status code: 200
Response text: [["NAME","B02001_001E","B02001_002E","B02001_003E","B02001_004E","B02001_005E","B02001_006E","B02001_007E","B02001_008E","B02001_009E","state","county","tract"],
["Census Tract 215.02, Bronx County, New York","6000","340","1515","0","67","0","3868","210","68","36","005","021502"],
["Census Tract 216.01, Bronx County, New York","3684","830","757","0","221","0","1793","83","17","36","005","021601"],
["Census Tract 216.02, Bronx County, New York","5636","694","2519","60","945","0","1249","169","60"
Status code: 200
Response text: [["NAME","B02001_001E","B02001_002E","B02001_003E","B02001_004E","B02001_005E","B02001_006E","B02001_007E","B02001_008E","B02001_009E","state","county","tract"],
["Census Tract 406, Cayuga County, New York","3400","3214","128","0","8","0","49","1","0","36","011","040600"],
["Census Tract 407, Cayuga County, New York","3633","3553","0","0","0","0","26","54","52","36","011","040700"],
["Census Tract 408, Cayuga County, New York","466

In [340]:
race2010.head()

,NAME,B02001_001E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,B02001_008E,B02001_009E,state,county,tract
0,"Census Tract 215.02, Bronx County, New York",6000,340,1515,0,67,0,3868,210,68,36,005,021502
1,"Census Tract 216.01, Bronx County, New York",3684,830,757,0,221,0,1793,83,17,36,005,021601
2,"Census Tract 216.02, Bronx County, New York",5636,694,2519,60,945,0,1249,169,60,36,005,021602
3,"Census Tract 217, Bronx County, New York",5663,416,2572,0,66,0,2465,144,135,36,005,021700
4,"Census Tract 218, Bronx County, New York",6158,907,1643,35,241,0,3277,55,13,36,005,021800


In [332]:
list(race2020['NAME'].unique())

['Census Tract 406, Cayuga County, New York',
 'Census Tract 407, Cayuga County, New York',
 'Census Tract 408, Cayuga County, New York',
 'Census Tract 409, Cayuga County, New York',
 'Census Tract 410.01, Cayuga County, New York',
 'Census Tract 410.02, Cayuga County, New York',
 'Census Tract 411.01, Cayuga County, New York',
 'Census Tract 411.02, Cayuga County, New York',
 'Census Tract 412.01, Cayuga County, New York',
 'Census Tract 412.02, Cayuga County, New York',
 'Census Tract 413, Cayuga County, New York',
 'Census Tract 414, Cayuga County, New York',
 'Census Tract 415, Cayuga County, New York',
 'Census Tract 416, Cayuga County, New York',
 'Census Tract 417, Cayuga County, New York',
 'Census Tract 418, Cayuga County, New York',
 'Census Tract 421, Cayuga County, New York',
 'Census Tract 9902, Cayuga County, New York',
 'Census Tract 301, Chautauqua County, New York',
 'Census Tract 302, Chautauqua County, New York',
 'Census Tract 303, Chautauqua County, New York',
 'C

In [334]:
list(race2010['NAME'].unique())

['Census Tract 215.02, Bronx County, New York',
 'Census Tract 216.01, Bronx County, New York',
 'Census Tract 216.02, Bronx County, New York',
 'Census Tract 217, Bronx County, New York',
 'Census Tract 218, Bronx County, New York',
 'Census Tract 219, Bronx County, New York',
 'Census Tract 220, Bronx County, New York',
 'Census Tract 221.01, Bronx County, New York',
 'Census Tract 221.02, Bronx County, New York',
 'Census Tract 222, Bronx County, New York',
 'Census Tract 223, Bronx County, New York',
 'Census Tract 224.01, Bronx County, New York',
 'Census Tract 224.03, Bronx County, New York',
 'Census Tract 224.04, Bronx County, New York',
 'Census Tract 225, Bronx County, New York',
 'Census Tract 227.01, Bronx County, New York',
 'Census Tract 227.02, Bronx County, New York',
 'Census Tract 227.03, Bronx County, New York',
 'Census Tract 228, Bronx County, New York',
 'Census Tract 229.01, Bronx County, New York',
 'Census Tract 229.02, Bronx County, New York',
 'Census Tract 2

In [347]:
def clean_race(df, year):
    df = df.rename(columns={
        "B02001_001E": f"total_{year}",
        "B02001_002E": f"white_{year}",
        "B02001_003E": f"black_{year}",
        "B02001_004E": f"native_{year}",
        "B02001_005E": f"asian_{year}",
        "B02001_006E": f"pi_{year}",
        "B02001_007E": f"other_{year}"
    })
    df = df.drop(columns=["B02001_008E", "B02001_009E"], errors="ignore")

    return df


race2010 = clean_race(race2010, 2010)
race2020 = clean_race(race2020, 2020)


In [348]:
race2010

,NAME,total_2010,white_2010,black_2010,native_2010,asian_2010,pi_2010,other_2010,state,county,tract
0,"Census Tract 215.02, Bronx County, New York",6000,340,1515,0,67,0,3868,36,005,021502
1,"Census Tract 216.01, Bronx County, New York",3684,830,757,0,221,0,1793,36,005,021601
2,"Census Tract 216.02, Bronx County, New York",5636,694,2519,60,945,0,1249,36,005,021602
3,"Census Tract 217, Bronx County, New York",5663,416,2572,0,66,0,2465,36,005,021700
4,"Census Tract 218, Bronx County, New York",6158,907,1643,35,241,0,3277,36,005,021800
...,...,...,...,...,...,...,...,...,...,...,...
4914,"Census Tract 130, Westchester County, New York",5972,4292,343,0,316,0,1009,36,119,013000
4915,"Census Tract 131.02, Westchester County, New York",6178,5325,49,0,718,44,0,36,119,013102
4916,"Census Tract 131.03, Westchester County, New York",5305,4609,66,0,348,0,191,36,119,013103
4917,"Census Tract 131.04, Westchester County, New York",5953,5344,16,0,365,0,0,36,119,013104


In [349]:
def clean_hisp(df, year):
    df = df.rename(columns={
        "B03002_001E": f"hisp_total_{year}",
        "B03002_003E": f"hispanic_{year}"
    })
    return df

hisp2010 = clean_hisp(hisp2010, 2010)
hisp2020 = clean_hisp(hisp2020, 2020)


In [350]:
nta10 = pd.read_excel('data/nyc2010census_tabulation_equiv.xlsx', skiprows=3)
nta10 = nta10.dropna(subset=["2010 Census Tract"]).rename(columns={
    "2010 Census Bureau FIPS County Code": "county",
    "2010 Census Tract": "tract",
    "Neighborhood Tabulation Area (NTA)": "NTA",
    "Unnamed: 6": "NTA_name"
})
nta10["tract"] = nta10["tract"].astype(float).astype(int).astype(str).str.zfill(6)
nta10["county"] = nta10["county"].astype(float).astype(int).astype(str).str.zfill(3)
nta10["state"] = "36"
nta10 = nta10[["state", "county", "tract", "NTA", "NTA_name"]]

nta20 = pd.read_excel('data/nyc2020census_tract_nta_cdta_relationships.xlsx')
nta20 = nta20.rename(columns={
    "CountyFIPS": "county",
    "CT2020": "tract",
    "NTACode": "NTA",
    "NTAName": "NTA_name"
})
nta20["tract"] = nta20["tract"].astype(str).str.zfill(6)
nta20["county"] = nta20["county"].astype(str).str.zfill(3)
nta20["state"] = "36"
nta20 = nta20[["state", "county", "tract", "NTA", "NTA_name"]]


In [351]:
# --- Normalize merge keys for ACS and NTA ---
for df in [race2010, race2020, nta10, nta20]:
    df["state"] = df["state"].astype(str).str.zfill(2)
    df["county"] = df["county"].astype(str).str.zfill(3)
    df["tract"] = (
        df["tract"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.zfill(6)
    )


In [302]:
race2010.head()

,NAME,total_2010,white_2010,black_2010,native_2010,asian_2010,pi_2010,other_2010,state,county,tract
0,"Census Tract 215.02, Bronx County, New York",6000,340,1515,0,67,0,3868,36,005,021502
1,"Census Tract 216.01, Bronx County, New York",3684,830,757,0,221,0,1793,36,005,021601
2,"Census Tract 216.02, Bronx County, New York",5636,694,2519,60,945,0,1249,36,005,021602
3,"Census Tract 217, Bronx County, New York",5663,416,2572,0,66,0,2465,36,005,021700
4,"Census Tract 218, Bronx County, New York",6158,907,1643,35,241,0,3277,36,005,021800


In [283]:
nta10

,state,county,tract,NTA,NTA_name
1,36,005,031000,BX31,Allerton-Pelham Gardens
2,36,005,031200,BX31,Allerton-Pelham Gardens
3,36,005,031400,BX31,Allerton-Pelham Gardens
4,36,005,031600,BX31,Allerton-Pelham Gardens
5,36,005,031800,BX31,Allerton-Pelham Gardens
...,...,...,...,...,...
2164,36,085,016901,SI07,Westerleigh
2165,36,085,018701,SI07,Westerleigh
2166,36,085,018901,SI07,Westerleigh
2167,36,085,019700,SI07,Westerleigh


In [382]:
nta20[nta20['tract']=='031000']

,state,county,tract,NTA,NTA_name
287,36,005,031000,BX1103,Pelham Gardens


In [383]:
# NYC county FIPS codes
nyc_counties = ["005", "047", "061", "081", "085"]  # Bronx, Kings, New York, Queens, Richmond

race2010 = race2010[race2010["county"].isin(nyc_counties)]
race2020 = race2020[race2020["county"].isin(nyc_counties)]
hisp2010 = hisp2010[hisp2010["county"].isin(nyc_counties)]
hisp2020 = hisp2020[hisp2020["county"].isin(nyc_counties)]


In [384]:
race2010_nta = race2010.merge(nta10, on=["tract"], how="left")
race2020_nta = race2020.merge(nta20, on=["tract"], how="left")

hisp2010_nta = hisp2010.merge(nta10, on=["tract"], how="left")
hisp2020_nta = hisp2020.merge(nta20, on=["tract"], how="left")


In [385]:
area_to_nta = pd.read_csv("area_to_nta.csv")  # you create this manually once

In [386]:
# Step 2 — keep only NTAs that appear in area_to_nta.csv
valid_ntas = area_to_nta["NTA_name"].unique()

race2010_nta = race2010_nta[race2010_nta["NTA_name"].isin(valid_ntas)]
race2020_nta = race2020_nta[race2020_nta["NTA_name"].isin(valid_ntas)]

# Step 2b — attach NTA_name to each row
race2010_nta = race2010_nta.merge(
    area_to_nta["NTA_name"],
    on="NTA_name",
    how="left"
)

race2020_nta = race2020_nta.merge(
    area_to_nta["NTA_name"],
    on="NTA_name",
    how="left"
)


In [393]:
race2020

,NAME,total_2020,white_2020,black_2020,native_2020,asian_2020,pi_2020,other_2020,state,county,tract
45,"Census Tract 92.02, Kings County, New York",3453,1023,196,27,1037,0,1064,36,047,009202
46,"Census Tract 94.01, Kings County, New York",2293,173,173,0,1582,0,353,36,047,009401
47,"Census Tract 94.02, Kings County, New York",2746,471,38,0,1908,0,154,36,047,009402
48,"Census Tract 96, Kings County, New York",5858,2122,261,44,1993,0,1261,36,047,009600
49,"Census Tract 98, Kings County, New York",6021,923,292,0,2306,0,2262,36,047,009800
...,...,...,...,...,...,...,...,...,...,...,...
5406,"Census Tract 539, Kings County, New York",2517,2032,90,0,1,0,286,36,047,053900
5407,"Census Tract 542, Kings County, New York",4096,3715,57,0,134,0,92,36,047,054200
5408,"Census Tract 543, Kings County, New York",0,0,0,0,0,0,0,36,047,054300
5409,"Census Tract 544, Kings County, New York",3590,2784,0,0,678,0,71,36,047,054400


In [388]:
for df in [race2010_nta, race2020_nta]:
    for col in df.columns:
        if any(g in col for g in ["white", "black", "asian", "native", "pi", "other", "total"]):
            df[col] = pd.to_numeric(df[col], errors="coerce")


In [ ]:
for col in ["white_2010","black_2010","asian_2010","native_2010",
            "pi_2010","other_2010","total_2010"]:
    race2010_nta[col] = pd.to_numeric(race2010_nta[col], errors="coerce")

for col in ["white_2020","black_2020","asian_2020","native_2020",
            "pi_2020","other_2020","total_2020"]:
    race2020_nta[col] = pd.to_numeric(race2020_nta[col], errors="coerce")


In [394]:
# Keep 2010 and 2020 tract-level data side-by-side
race_tract = race2010.merge(race2020, on=["state", "county", "tract"])
# Calculate changes at tract level

In [397]:
# Merge both 2010 and 2020 race data to 2020 NTA
race2010_to_2020nta = race2010.merge(nta20, on=["state", "county", "tract"])
race2020_to_2020nta = race2020.merge(nta20, on=["state", "county", "tract"])
groups = ["white", "black", "asian", "native", "pi", "other"]
race_cols = (
    [f"{g}_2010" for g in groups] + 
    ["total_2010"] +
    [f"{g}_2020" for g in groups] + 
    ["total_2020"]
)

agg_2010 = race2010_to_2020nta.groupby("NTA_name")[
    [f"{g}_2010" for g in groups] + ["total_2010"]
].sum()

agg_2020 = race2020_to_2020nta.groupby("NTA_name")[
    [f"{g}_2020" for g in groups] + ["total_2020"]
].sum()

# Aggregate by NTA
agg_2010 = race2010_to_2020nta.groupby("NTA_name")[race_cols].sum()
agg_2020 = race2020_to_2020nta.groupby("NTA_name")[race_cols].sum()

KeyError: "Columns not found: 'other_2020', 'total_2020', 'pi_2020', 'asian_2020', 'white_2020', 'native_2020', 'black_2020'"

In [390]:
# # Shorten 2020 NTA codes to first 4 characters
# race2020_nta["NTA_short"] = race2020_nta["NTA"].str[:4]
# race2010_nta["NTA_short"] = race2010_nta["NTA"]


In [391]:
groups = ["white", "black", "asian", "native", "pi", "other"]

agg2010 = race2010_nta.groupby("NTA_name", as_index=False).agg(
    {f"{g}_2010": "sum" for g in groups} | {"total_2010": "sum"}
)

agg2020 = race2020_nta.groupby("NTA_name", as_index=False).agg(
    {f"{g}_2020": "sum" for g in groups} | {"total_2020": "sum"}
)


In [392]:
agg2010

,NTA_name,white_2010,black_2010,asian_2010,native_2010,pi_2010,other_2010,total_2010
0,Allerton-Pelham Gardens,1454528,1074880,290880,7488,256,467008,3420864
1,Astoria,13349248,2406912,1814720,50624,3904,2440896,20643648
2,Auburndale,654080,16320,511616,0,0,59264,1254592
3,Bath Beach,2211712,1155712,764096,12160,5312,980736,5266048
4,Battery Park City-Lower Manhattan,8848640,1304576,2427904,24064,0,622080,13640448
...,...,...,...,...,...,...,...,...
95,West Village,7913600,1586112,1019648,29504,1856,1437376,12293056
96,Whitestone,1592960,14208,348352,896,0,58240,2042816
97,Williamsbridge-Olinville,1944896,4165376,730688,37824,2752,682880,7729920
98,Windsor Terrace,2400640,596160,297856,15232,0,237440,3630144


In [380]:
race_nta = agg2010.merge(agg2020, on="NTA_name", how="outer")


In [381]:
race_nta

,NTA_name,white_2010,black_2010,asian_2010,native_2010,pi_2010,other_2010,total_2010,white_2020,black_2020,asian_2020,native_2020,pi_2020,other_2020,total_2020
0,Allerton,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2952768.0,4938816.0,578304.0,29952.0,7872.0,2796864.0,11933760.0
1,Allerton-Pelham Gardens,1454528.0,1074880.0,290880.0,7488.0,256.0,467008.0,3420864.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Astoria,13349248.0,2406912.0,1814720.0,50624.0,3904.0,2440896.0,20643648.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Astoria (Central),NaN,NaN,NaN,NaN,NaN,NaN,NaN,4723200.0,1195200.0,783040.0,43840.0,576.0,985600.0,8289152.0
4,Astoria (East)-Woodside (North),NaN,NaN,NaN,NaN,NaN,NaN,NaN,7079616.0,2198400.0,1081088.0,32320.0,2176.0,1603904.0,12731136.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,West Village,7913600.0,1586112.0,1019648.0,29504.0,1856.0,1437376.0,12293056.0,3984256.0,1078528.0,496768.0,35200.0,32576.0,1136640.0,7367104.0
149,Whitestone,1592960.0,14208.0,348352.0,896.0,0.0,58240.0,2042816.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,Williamsbridge-Olinville,1944896.0,4165376.0,730688.0,37824.0,2752.0,682880.0,7729920.0,1628352.0,3777664.0,688000.0,43648.0,11520.0,907840.0,7356864.0
151,Windsor Terrace,2400640.0,596160.0,297856.0,15232.0,0.0,237440.0,3630144.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [322]:
area_to_nta = pd.read_csv("area_to_nta.csv")  # you create this manually once


In [323]:
# Step 1: Filter to NYC counties
nyc_counties = ["005", "047", "061", "081", "085"]
race2010 = race2010[race2010["county"].isin(nyc_counties)]
race2020 = race2020[race2020["county"].isin(nyc_counties)]

# Step 2: Merge ACS with NTA lookup
race2010_nta = race2010.merge(nta10, on=["state", "county", "tract"], how="left")
race2020_nta = race2020.merge(nta20, on=["state", "county", "tract"], how="left")

# Step 3: Aggregate by NTA_name (not NTA code)
groups = ["white", "black", "asian", "native", "pi", "other"]

agg2010 = race2010_nta.groupby("NTA_name", as_index=False).agg(
    {f"{g}_2010": "sum" for g in groups} | {"total_2010": "sum"}
)

agg2020 = race2020_nta.groupby("NTA_name", as_index=False).agg(
    {f"{g}_2020": "sum" for g in groups} | {"total_2020": "sum"}
)

# Step 4: Merge 2010 and 2020 data by NTA_name
race_nta = agg2010.merge(agg2020, on="NTA_name", how="outer")

# Step 5: Merge with area_to_nta.csv to ensure rent alignment
race_nta = race_nta.merge(area_to_nta[["NTA_name", "NTA"]], on="NTA_name", how="left")


In [316]:
agg2010

,NTA_short,NTA_name,white_2010,black_2010,asian_2010,native_2010,pi_2010,other_2010,total_2010
0,BK09,Brooklyn Heights-Cobble Hill,18871,1187,1492,90,10,639,23255
1,BK17,Sheepshead Bay-Gerritsen Beach-Manhattan Beach,45718,3844,10306,0,0,1192,61450
2,BK19,Brighton Beach,24479,379,4208,161,0,1014,30771
3,BK21,Seagate-Coney Island,13403,11941,2193,72,17,2670,30992
4,BK23,West Brighton,15507,212,159,0,0,36,15959
...,...,...,...,...,...,...,...,...,...
190,SI37,Stapleton-Rosebank,13657,4727,3362,116,0,2385,24747
191,SI45,New Dorp-Midland Beach,18653,463,1395,66,0,1125,21907
192,SI48,Arden Heights,22217,408,1804,19,0,275,24853
193,SI54,Great Kills,38630,63,1489,211,0,395,41033


In [320]:
agg2020['NTA_short'][:10]

0    BK01
1    BK02
2    BK03
3    BK04
4    BK05
5    BK06
6    BK07
7    BK08
8    BK09
9    BK10
Name: NTA_short, dtype: object

In [326]:
race2010.head()

,NAME,total_2010,white_2010,black_2010,native_2010,asian_2010,pi_2010,other_2010,state,county,tract
0,"Census Tract 215.02, Bronx County, New York",6000,340,1515,0,67,0,3868,36,005,021502
1,"Census Tract 216.01, Bronx County, New York",3684,830,757,0,221,0,1793,36,005,021601
2,"Census Tract 216.02, Bronx County, New York",5636,694,2519,60,945,0,1249,36,005,021602
3,"Census Tract 217, Bronx County, New York",5663,416,2572,0,66,0,2465,36,005,021700
4,"Census Tract 218, Bronx County, New York",6158,907,1643,35,241,0,3277,36,005,021800


In [327]:
race2020.head()

,NAME,total_2020,white_2020,black_2020,native_2020,asian_2020,pi_2020,other_2020,state,county,tract
45,"Census Tract 92.02, Kings County, New York",3453,1023,196,27,1037,0,1064,36,047,009202
46,"Census Tract 94.01, Kings County, New York",2293,173,173,0,1582,0,353,36,047,009401
47,"Census Tract 94.02, Kings County, New York",2746,471,38,0,1908,0,154,36,047,009402
48,"Census Tract 96, Kings County, New York",5858,2122,261,44,1993,0,1261,36,047,009600
49,"Census Tract 98, Kings County, New York",6021,923,292,0,2306,0,2262,36,047,009800


In [324]:
race_nta

,NTA_name,white_2010,black_2010,asian_2010,native_2010,pi_2010,other_2010,total_2010,white_2020,black_2020,asian_2020,native_2020,pi_2020,other_2020,total_2020,NTA
0,Airport,00,00,00,00,00,00,00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Allerton,NaN,NaN,NaN,NaN,NaN,NaN,NaN,653945278826509309773601192186,82865012716901046200210732138941,50226123124791764529556,0720019016490,5000000360,1144175411121747134215225591778446,328639895456530338215121226156831698,BX1104
2,Allerton,NaN,NaN,NaN,NaN,NaN,NaN,NaN,653945278826509309773601192186,82865012716901046200210732138941,50226123124791764529556,0720019016490,5000000360,1144175411121747134215225591778446,328639895456530338215121226156831698,BX1104
3,Allerton,NaN,NaN,NaN,NaN,NaN,NaN,NaN,653945278826509309773601192186,82865012716901046200210732138941,50226123124791764529556,0720019016490,5000000360,1144175411121747134215225591778446,328639895456530338215121226156831698,BX1104
4,Allerton,NaN,NaN,NaN,NaN,NaN,NaN,NaN,653945278826509309773601192186,82865012716901046200210732138941,50226123124791764529556,0720019016490,5000000360,1144175411121747134215225591778446,328639895456530338215121226156831698,BX1104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12891,park-cemetery-etc-Bronx,090000420,063377002270,0000000,0000000,0000000,01200001250,088977004160,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12892,park-cemetery-etc-Brooklyn,0220000114013700,00000000900,00000000000,00000000000,00000000000,00000000000,0220000127014600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12893,park-cemetery-etc-Manhattan,5770000014,1637000000,47000000,0000000,0000000,130000000,25070000014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12894,park-cemetery-etc-Queens,000550750000800000,0000000000000000,005900000000190000,0000000000000000,0000000000000000,0000000000000000,00595507500008190000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
